In [ ]:
# Tải unsloth
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm==0.8.5.post1

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
# Tải các thư viện cần thiết
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm==0.8.5.post1
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [ ]:
from unsloth import FastModel, FastLanguageModel,UnslothTrainer, UnslothTrainingArguments
import torch
print("200 OK")

In [ ]:
# Thư viện cho LLM
from transformers import (
    AutoModelForCausalLM, # Tinh chỉnh mô hình trả lời nhân quả
    AutoTokenizer, # Token hóa câu văn
    BitsAndBytesConfig, # điều chỉnh hiệu quả đông bộ nhớ
    HfArgumentParser,
    TrainingArguments, # Quản lý tham số,
    pipeline,
    logging # ghi lại thông tin loss, accuracy
)

# Tinh chỉnh adapter
from peft import (
    LoraConfig, # Cấu hình cho lora
    PeftModel, # Lưu trữ pp fine tune hiệu quả
    prepare_model_for_kbit_training, # model để training
    get_peft_model # lấy ra sau khi fineTune
)

import os, torch, wandb # wandb giúp lưu trư các thông tin trong quá trình train và vẽ biểu đồ
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format # Thực hiện training bằng supervised
import bitsandbytes as bnb
print("200 OK")

In [ ]:
# Đăng nhập vào wandb
from google.colab import userdata
import os
wandb_token = os.getenv("WANDB_API_KEY") or userdata.get("WANDB_API_KEY")
if not wandb_token:
    raise ValueError("Missing WANDB_API_KEY. Set env var or add to Colab secrets.")
wandb.login(key=wandb_token)

In [ ]:
# Test tập dữ liệu
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from huggingface_hub import login, create_repo # thư viện để login dựa vào api key
# Load tập train
dataset = load_dataset("Thang21/chatdocter", split="train")
print(dataset)
print(f"Số lượng hàng của dataset sau khi load: {dataset.num_rows}")

In [ ]:
# Sử dụng tập dữ liệu để huấn luyện
# Chia thành 20 phần (Mỗi lần tăng thêm 1 lần lấy)
datasets = [dataset.shard(num_shards=20, index=i) for i in range(20)]
test_data = datasets[15]
train_val_dict = test_data.train_test_split(test_size=0.2, seed = 42)
# Tập trains
train_data = train_val_dict['train']
# Tập val
cross_data = train_val_dict['test']
print(f"train length {train_data.num_rows}")
print(f"cross length {cross_data.num_rows}")

In [ ]:
# Lấy mô hình và gắn adapter có sẵn
max_seq_length = 512
# Lấy mô hình và adapter có sẵn
model,tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    #device_map = "auto"
)
model = PeftModel.from_pretrained(
    model,
    "PhongGoldFish/testing_finetune_14",  # repo Hugging Face hoặc thư mục local # Phải thay đổi adapter nữa
    is_trainable = True  # Nếu bạn muốn fine-tune tiếp
)
tokenizer.chat_template ="""
{% for message in messages %}
{% if message['role'] == 'system' %}
<|start_of_turn|><|system|>
{{ message['content'] }}<|end_of_turn|>
{% elif message['role'] == 'user' %}
<|start_of_turn|><|user|>
{{ message['content'] }}<|end_of_turn|>
{% elif message['role'] == 'assistant' %}
<|start_of_turn|><|assistant|>
{{ message['content'] }}<|end_of_turn|>
{% endif %}
{% endfor %}
""".strip()
print("200 Ok")

In [ ]:
### có batch

system_prompt = """Bạn là một trợ lý ảo và tư vấn viên chuyên nghiệp cho người Việt Nam.
Nhiệm vụ của bạn là trả lời tất cả câu hỏi liên quan đến kiến thức chung (khoa học, xã hội, đời sống, giáo dục, v.v.) bằng tiếng Việt có thể dùng thuật ngữ tiếng Anh.
Bạn cần sử dụng ngôn ngữ dễ hiểu, văn phong lịch sự, gần gũi, và luôn thể hiện sự tận tình, chu đáo trong từng câu trả lời."""

def format_data_for_sft(example, tokenizer, system_prompt):
    """
    Định dạng một ví dụ dữ liệu thành một chuỗi văn bản duy nhất
    theo chat template của tokenizer.

    Hàm này sẽ được dùng trong dataset.map()
    """
    messages = []

    # 1. Thêm tin nhắn của người dùng
    user_message_content = ""
    if example.get('input') and example['input'].strip():
        user_message_content = f"{system_prompt}\n\nCâu hỏi: {example['instruction']}\nThông tin thêm: {example['input']}"
    else:
        user_message_content = f"{system_prompt}\n\nCâu hỏi: {example['instruction']}"

    messages.append({"role": "user", "content": user_message_content})

    # 2. Thêm câu trả lời của trợ lý ảo (model)
    messages.append({"role": "assistant", "content": example['output']})

    # 3. Sử dụng hàm apply_chat_template của tokenizer để tạo chuỗi văn bản cuối cùng.
    #    Hàm này sẽ tự động thêm các token đặc biệt như <start_of_turn>, v.v.
    #    tokenize=False: chỉ trả về string, không trả về input_ids.
    #    add_generation_prompt=False: không thêm prompt để model sinh tiếp, vì ta có cả câu trả lời.
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # 4. Trả về một dictionary với một cột duy nhất là "text"
    return { "text": formatted_text }

formatted_train_data = train_data.map(
    lambda example: format_data_for_sft(example, tokenizer, system_prompt),
    num_proc=4 # Sử dụng nhiều tiến trình để tăng tốc
)

formatted_cross_data = cross_data.map(
    lambda example: format_data_for_sft(example, tokenizer, system_prompt),
    num_proc=4
)

# In ra một ví dụ để kiểm tra
print("Ví dụ dữ liệu sau khi định dạng:")
print(formatted_train_data[0]['text'])


In [ ]:
# Kiểm tra layer và các thông số của adapter
for name, module in model.named_modules():
    if hasattr(module, 'lora_A') and hasattr(module, 'lora_B'):
        try:
            # Nếu là ModuleDict thì lấy 'default'
            lora_A = module.lora_A["default"]
            lora_B = module.lora_B["default"]
        except (TypeError, KeyError):
            # Nếu không phải dict, dùng trực tiếp
            lora_A = module.lora_A
            lora_B = module.lora_B

        print(f"\n=== Adapter Layer: {name} ===")
        print(f"lora_A shape: {lora_A.weight.shape}")
        print(f"lora_B shape: {lora_B.weight.shape}")

        print(f"First row lora_A:\n{lora_A.weight[0].detach().cpu().numpy()}")
        print(f"First row lora_B:\n{lora_B.weight[0].detach().cpu().numpy()}")

        # In thêm hệ số nếu có
        if hasattr(module, 'scaling'):
            print(f"scaling: {module.scaling}")
        if hasattr(module, 'lora_alpha'):
            print(f"lora_alpha: {module.lora_alpha}")

In [ ]:
# Đích lưu adapter trên huggingface
output_model = 'PhongGoldFish/testing_finetune_15'
from transformers import TrainingArguments
from trl import SFTTrainer
from huggingface_hub import login # thư viện để login dựa vào api key
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Qlora")

# login vào huggingface
login(token = secret_value_0)

# Đây là cấu hình TrainingArguments đã được tối ưu cho Unsloth
#TrainingArguments
training_arguments = UnslothTrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size = 2, # Thử tăng lên 4 với Unsloth
    per_device_eval_batch_size = 4,  # Đồng bộ
    gradient_accumulation_steps = 2, # Giảm xuống để effective batch size = 16

    optim = "paged_adamw_8bit", # Unsloth khuyến nghị 8bit
    num_train_epochs = 2,

    # Sử dụng warmup_ratio thay cho steps để linh hoạt hơn
    warmup_ratio = 0.1,
    weight_decay=0.01,

    # --- PHẦN TỰ ĐỘNG CHỌN ĐỊNH DẠNG ---
    # Kiểm tra xem GPU có hỗ trợ bfloat16 không
    bf16 = torch.cuda.is_bf16_supported(),
    # Nếu không hỗ trợ bf16, thì sử dụng fp16
    fp16 = not torch.cuda.is_bf16_supported(),
    # ------------------------------------

    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100, # Đánh giá thường xuyên hơn để kiểm tra

    learning_rate = 1e-5,
    lr_scheduler_type = "linear",

    report_to="wandb",
)

# đổi 3 thứ : output_model, adapter dir, position of dataset
trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_train_data,
    eval_dataset = formatted_cross_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    num_train_epochs = 3,
    args = training_arguments
)

# Bây giờ bạn có thể bắt đầu training
trainer.train()
wandb.finish()

local_dir = "/kaggle/working/saved_model"
trainer.model.save_pretrained(local_dir)

trainer.model.push_to_hub(repo_id = output_model)
print("200 OK")